# Demo: GNN-BERT Music Context Understanding

One end-to-end inference example (Task 3 fusion model, trained on real
MagnaTagATune data): given a track's structure graph + a text caption,
predict tags. (Note: valence/arousal are not applicable here -- MTAT
has no emotion labels; see the DEAM-trained variant for that.)

Requires: real MTAT graphs unpacked to /content/mtat_graphs/ (via
`python src/unpack_dir.py --in_file data/packed/mtat_graphs.npz --out_dir /content/mtat_graphs`)
and the real checkpoint at results/task3_fusion_mtat.pt.

In [3]:
import sys, json
sys.path.insert(0, 'src')
import torch
import numpy as np
import yaml
from transformers import AutoTokenizer
from fusion_model import GNNBertFusionModel
from datasets import load_tag_vocab, load_manifest

with open('config_mtat.yaml') as f:
    cfg = yaml.safe_load(f)
tags = load_tag_vocab(toy_dir='data/processed/toy')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
tokenizer = AutoTokenizer.from_pretrained(cfg['model']['bert_name'])
test_items = load_manifest('test', splits_dir='data/splits')
example = test_items[0]
graph = np.load(f"/content/mtat_graphs/{example['track_id']}.npz")
in_dim = graph['seg_x'].shape[1]

model = GNNBertFusionModel(
    bert_name=cfg['model']['bert_name'], gnn_in_dim=in_dim,
    gnn_hidden=cfg['model']['gnn_hidden'], gnn_layers=cfg['model']['gnn_layers'],
    num_tags=len(tags), fusion_type=cfg['model']['fusion_type'],
).to(device)
model.load_state_dict(torch.load('results/task3_fusion_mtat.pt', map_location=device))
model.eval()

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GNNBertFusionModel(
  (text_encoder): BertTextEncoder(
    (bert): DistilBertModel(
      (embeddings): Embeddings(
        (word_embeddings): Embedding(30522, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (transformer): Transformer(
        (layer): ModuleList(
          (0-5): 6 x TransformerBlock(
            (attention): DistilBertSelfAttention(
              (q_lin): Linear(in_features=768, out_features=768, bias=True)
              (k_lin): Linear(in_features=768, out_features=768, bias=True)
              (v_lin): Linear(in_features=768, out_features=768, bias=True)
              (out_lin): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (ffn): FFN(
   

In [5]:
enc = tokenizer(example['caption'], truncation=True, padding='max_length', max_length=cfg['data']['max_text_len'], return_tensors='pt')
x = torch.tensor(graph['seg_x'], dtype=torch.float32)
edge_index = torch.tensor(graph['seg_edge_index'], dtype=torch.long)
batch_index = torch.zeros(x.shape[0], dtype=torch.long)

with torch.no_grad():
    tag_logits, va_pred = model(
        enc['input_ids'].to(device), enc['attention_mask'].to(device),
        x.to(device), edge_index.to(device), batch_index.to(device),
    )
    probs = torch.sigmoid(tag_logits).cpu().numpy()[0]

print('Track:', example['track_id'])
print('Caption (input):', example['caption'])
print('True tags:', [t for t, v in zip(tags, example['tags']) if v])
print('Predicted tags (p>0.5):', [(t, round(float(p), 3)) for t, p in zip(tags, probs) if p > 0.5])
print('(Valence/arousal not applicable: MTAT has no emotion labels)')

Track: mtat_2
Caption (input): Tags: classical.
True tags: ['classical', 'strings', 'violin', 'opera']
Predicted tags (p>0.5): [('classical', 0.978)]
(Valence/arousal not applicable: MTAT has no emotion labels)
